# BAC-CAB via automatic rewrite search

The **BAC-CAB rule** applies when the cross product nests as $\mathbf{a} \times (\mathbf{b} \times \mathbf{c})$.
When the nesting is reversed — $(\mathbf{v} \times \mathbf{w}) \times \mathbf{u}$ — the identity does not
match directly.  `search_apply` finds the preparatory rewrite (anti-commutativity) and applies
BAC-CAB automatically.

This notebook demonstrates:
1. `search_apply` on the direct case (one step, no preparation needed).
2. `search_apply` on the swapped case (anti-commutativity first, then BAC-CAB).
3. The full derivation history rendered as LaTeX.

> **Prerequisite:** build the Python bindings and set `PYTHONPATH`, or run
> `source examples/env.sh` before launching Jupyter.

In [ ]:
import pathlib
from tender import (
    tensor,
    cross,
    State,
    Derivation,
    show_jupyter,
    search_apply,
    to_latex_document,
)
from tender.lib.identities.epsilon import bac_cab, anti_commutativity

In [ ]:
u = tensor(r"\mathbf{u}", 1)
v = tensor(r"\mathbf{v}", 1)
w = tensor(r"\mathbf{w}", 1)

## Direct case: $\mathbf{u} \times (\mathbf{v} \times \mathbf{w})$

BAC-CAB matches at the root — `search_apply` returns a single step.

In [ ]:
expr_direct = cross(u, cross(v, w))
steps_direct = search_apply(bac_cab, expr_direct)
print(f"{len(steps_direct)} step(s): {[s.name for s in steps_direct]}")
history_direct = Derivation(steps_direct).apply(State(expr_direct))
show_jupyter(history_direct)

## Swapped case: $(\mathbf{v} \times \mathbf{w}) \times \mathbf{u}$

The nesting order is wrong for BAC-CAB.  `search_apply` discovers that applying
cross-product **anti-commutativity** first rearranges the expression into a form
where BAC-CAB matches, and returns both steps.

In [ ]:
expr_swapped = cross(cross(v, w), u)
steps_swapped = search_apply(bac_cab, expr_swapped)
print(f"{len(steps_swapped)} step(s): {[s.name for s in steps_swapped]}")
history_swapped = Derivation(steps_swapped).apply(State(expr_swapped))
show_jupyter(history_swapped)

## Verification

Both expressions describe the same geometric object, so their expansions should
be equal up to sign (the swapped form picks up a global minus from anti-commutativity).

In [ ]:
assert len(steps_direct) == 1
assert len(steps_swapped) == 2
assert steps_swapped[0].name == "apply(cross-anticomm)"
assert steps_swapped[1].name == "apply(BAC-CAB)"
print("All assertions passed.")

## Export to PDF

Write a compilable LaTeX document for the swapped-case derivation.

In [ ]:
tex = to_latex_document(history_swapped, title="BAC-CAB via anti-commutativity")
out_dir = pathlib.Path("out")
out_dir.mkdir(exist_ok=True)
(out_dir / "bac_cab_search.tex").write_text(tex)
print("Written to out/bac_cab_search.tex")
print("Compile with: pdflatex -output-directory out out/bac_cab_search.tex")